**Load flow exercise – Correction**

This project requires Python 3.7 or above, to check this, we can do as follows:

In [35]:
import sys
assert sys.version_info >= (3, 7)

In [36]:
import os
import sys
# project_name =  "git/rht"
# project_folder = os.path.join(os.path.expanduser('~'), project_name)
# print(project_folder)
# sys.path.append(project_folder)
# print("We are here",os.getcwd())
# os.chdir("corrections")
os.chdir(os.getcwd().replace("/tutorials", ""))
# print("We are here",os.getcwd())
!pwd

/home/thierryfra/github/pro/rht


Before to start anything, we should add the necessary packages, as follows:

In [37]:
import pandapower as pp
import pandas as pd
import numpy as np
import pp_heig_plot as pp_plot
import pp_heig_simulation as pp_sim
from datetime import time
from numpy.linalg import inv
from pp_heig_simulation import load_net_from_xlsx

# Exercise power flow

In this exercise, we will dive into the power flow analysis of a 5-mesh electrical network topology (figure below).

The objective is to understand and implement a complete robust power flow algorithm, bringing clarity to complex electricity grid calculations. Refer to the figure below for a visual representation of a 5-mesh network.

Checklist of the exercise:
1. `cartesian2z` and `polar2z`
    - Transform coordinates into complex numbers.
2. `convert_line_data_in_pu`: 
    - Fill line_pu DataFrame missing columns.
3. `compute_admittance_matrix`
    - Calculate admittance matrix elements.
4. `update_bus_power`: 
    - Decompose complex voltages and admittance matrix.
    - Update bus powers.
5. `jacobian_matrix`: 
    - Decompose complex voltages and admittance matrix.
    - Calculate active and reactive power partial differential depending on voltages modulus and angle.
6. `update_voltage`:     
    - Calculate state variable delta.
    - Update voltages polars coordinates.
    - Build complex voltages.
7. `process_results`:     
    - For each line find parameters: from_bus, to_bus, longitudinal admittance and transversal susceptance.
    - Calculate line current in pu.
    - Calculate line apparent power in pu.
8. Power flow algorithm
    - Define grid base values
    - Define slack bus index.
    - Implement algorithm steps 5 to 8.
  
You'll have to go through all the functions and code the part we're asking you to do.

<!-- <img alt="5-mesh power network" width="600" caption="5-mesh power network" src="./exercises/5-mesh_electrical_power_grid.png" id="5-mesh_grid"/>
<a href="./exercises/5-mesh_electrical_power_grid.png"></a> -->

<!-- *Figure – 5-mesh electrical power grid* -->

# Initial parameters and table based on the 5-mesh grid

Nominal values:

In [38]:
V_base = 400 
S_base = 1e5
Z_base = V_base ** 2 / S_base
f_base = 50

Define algorithm parameters:

In [39]:
tol_limit = 1e-4
iter_max = 10

Load data in **panda**power format:

In [40]:
file_path = "./corrections/5_meshed_bus.xlsx"
net = load_net_from_xlsx(file_path=file_path)

/home/thierryfra/miniforge3/envs/nbdev/lib/python3.12/site-packages/pp_heig_simulation.py:96: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/home/thierryfra/miniforge3/envs/nbdev/lib/python3.12/site-packages/pp_heig_simulation.py:96: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/home/thierryfra/miniforge3/envs/nbdev/lib/python3.12/site-packages/pp_heig_simulation.py:96: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to 

First things first, we check the network in the Excel table named `5-mesh_grid_example` with the following function:

In [41]:
pp_plot.plot_power_network(net=net, plot_title="5-mesh power grid example", filename="5-mesh_grid_example")

We can now quickly look at what's inside this dictionnary of DataFrames:

In [42]:
net

This pandapower network includes the following parameter tables:
   - bus (5 elements)
   - load (4 elements)
   - sgen (2 elements)
   - ext_grid (1 element)
   - line (6 elements)

In [43]:
net["bus"]

,name,vn_kv,type,zone,in_service
0,bus_1,0.4,b,PQ-load_1,True
1,bus_2,0.4,b,PQ-load_2,True
2,bus_3,0.4,b,PQ_load_3,True
3,bus_4,0.4,b,PQ_load_4,True
4,bus_5,0.4,b,PQ_gen_5,True


In [44]:
net["bus"].info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        5 non-null      object 
 1   vn_kv       5 non-null      float64
 2   type        5 non-null      object 
 3   zone        5 non-null      object 
 4   in_service  5 non-null      bool   
dtypes: bool(1), float64(1), object(3)
memory usage: 377.0+ bytes


In [45]:
net["line"]

,name,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,max_i_ka,parallel,type,r0_ohm_per_km,x0_ohm_per_km,c0_nf_per_km,g_us_per_km,df,std_type,in_service
0,line_0,0,1,10.0,0.006000,0.017300,0.0,0.627,1,None,0.0,0.0,0.0,0.0,1.0,None,True
1,line_1,0,3,15.0,0.009909,0.002924,27.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True
2,line_2,0,4,7.0,0.001190,0.000510,5.5,0.380,1,None,0.0,0.0,0.0,0.0,1.0,None,True
3,line_3,1,2,9.0,0.006576,0.002087,20.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True
4,line_4,2,3,20.0,0.006000,0.017300,0.0,0.627,1,None,0.0,0.0,0.0,0.0,1.0,None,True
5,line_5,3,4,8.0,0.009909,0.002924,27.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True


What we're doing here is navigating through the various tabs of an Excel spreadsheet...

In [46]:
net["line"].info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 5
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6 non-null      object 
 1   from_bus       6 non-null      int64  
 2   to_bus         6 non-null      int64  
 3   length_km      6 non-null      float64
 4   r_ohm_per_km   6 non-null      float64
 5   x_ohm_per_km   6 non-null      float64
 6   c_nf_per_km    6 non-null      float64
 7   max_i_ka       6 non-null      float64
 8   parallel       6 non-null      int64  
 9   type           0 non-null      object 
 10  r0_ohm_per_km  6 non-null      float64
 11  x0_ohm_per_km  6 non-null      float64
 12  c0_nf_per_km   6 non-null      float64
 13  g_us_per_km    6 non-null      float64
 14  df             6 non-null      float64
 15  std_type       0 non-null      object 
 16  in_service     6 non-null      bool   
dtypes: bool(1), float64(10), int64(3), object(3)
memory usage: 994.

In [47]:
net["sgen"]

,name,bus,p_mw,q_mvar,sn_mva,scaling,profile_mapping,type,k,rx,current_source,in_service
0,pv_0,1,0.1,0.0,None,1.0,-1,None,None,None,True,True
1,pv_1,4,0.2,0.0,None,1.0,-1,None,None,None,True,True


In this DataFrame, one can observe where the gen are.

We can easily see the different features of pandas library that we have seen in through [Pandas Quick Start Guide](./tutorials/03_pandas.ipynb).

## Used functions
### Remove slack bus

In [48]:
def remove_slack_bus(data_np: np.array, slack_bus: int) -> np.array:
    """Remove data associated with the slack bus from a 1 or 2-dimensional NumPy array.
    
    Parameters
    ----------
    data_np : np.array 
        The NumPy array of the data to be processed.
    slack_bus : int
        The index of the slack bus to be removed.

    Returns
    -------
    numpy.ndarray
        A new NumPy array with the data related to the slack bus removed

    Raises
    ------
    ValueError
        If `data_np` is not a NumPy array, or if the `slack_bus` index is out of bounds.

    Notes
    -----
    This function removes data associated with the slack bus, specified by its index, from a 1 or 2-dimensional NumPy array.
    """
    if len(data_np.shape) == 2:
        return np.delete(np.delete(data_np, slack_bus, axis=0), slack_bus, axis=1)
    elif data_np.ndim == 1:
        return np.delete(data_np, slack_bus)
    else:
        return None

### Convert cartesian and polar coordinate to complex numbers

For polar coordinates, please refer to RHT RE04-5 course, slide 27.

In [49]:
def cartesian2z(real: np.array, imag: np.array) -> np.array:
    """Convert cartesian coordinates to a complex number.

    Parameters
    ----------
    real : float
        An array of complex numbers' real part.
    imag : float
        An array of complex numbers' imaginary part.

    Returns
    -------
    complex
        A complex number represented in rectangular coordinates.
    """
    return real + 1j * imag

In [50]:
def polar2z(module: np.array, angle: np.array) -> np.array:
    """Convert polar coordinates to a complex number.

    Parameters
    ----------
    module : np.array
        An array of complex numbers' magnitude (module).
    angle : np.array
        An array of complex numbers' angle (in radians).

    Returns
    -------
    np.array
        An array of complex number
    """
    return module * np.exp(1j * angle)

### Convert line parameters in pu

For the per-unit system, please refer to RHT RE04-05 course, slides 14–17.

In [51]:
def convert_line_data_in_pu(net: pp.pandapowerNet, Z_base: float, f_base: float) -> pd.DataFrame:
    """Convert line data from per unit (p.u.) system to a Pandas DataFrame.

    Parameters
    ----------
    net : pandapowerNet
        The pandapower network for which line data needs to be converted to p.u.
    Z_base : float
        The base impedance value in ohms for the p.u. system.
    f_base : float
        The system frequency in Hertz.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing line data in per unit (p.u.) system.
    """
    lines_pu: pd.DataFrame = pd.DataFrame(
        columns=["from_bus", "to_bus", "r_pu", "x_pu", "b_pu", "z_pu", "y_pu"]
        )
    
    # Define lines' bus connections.
    lines_pu = net.line[["from_bus", "to_bus"]]
    # Convert line longitudinal resistance from ohm/km in p.u.
    lines_pu["r_pu"] =  net.line["r_ohm_per_km"]*net.line["length_km"] / Z_base
    # Convert line longitudinal inductance from ohm/km in p.u.
    lines_pu["x_pu"] =  net.line["x_ohm_per_km"]*net.line["length_km"] / Z_base
    # Convert line capacitors from nF/km to p.u.
    lines_pu["b_pu"] = net.line["c_nf_per_km"] * net.line["length_km"] * 2*np.pi* 1e-9* f_base*Z_base
    # Create line longitudinal impedance and admittance.                                                    
    lines_pu["z_pu"] = cartesian2z(lines_pu["r_pu"].values, lines_pu["x_pu"].values)
    lines_pu["y_pu"] = 1/lines_pu["z_pu"]    
    # Create line transversal susceptance. .
    lines_pu["b_pu"] = cartesian2z(np.zeros(lines_pu.shape[0]), lines_pu["b_pu"].values)
    return lines_pu
    

Once we get there, you can see the transformation of your initial "line" DataFrame into per unit values as follows:

In [52]:
df_pu = convert_line_data_in_pu(net=net, Z_base=Z_base, f_base=f_base)
df_pu

,from_bus,to_bus,r_pu,x_pu,b_pu,z_pu,y_pu
0,0,1,0.037500,0.108125,0.000000+0.000000j,0.037500+0.108125j,2.863193- 8.255540j
1,0,3,0.092897,0.027412,0.000000+0.000207j,0.092897+0.027412j,9.902372- 2.922044j
2,0,4,0.005206,0.002231,0.000000+0.000019j,0.005206+0.002231j,162.271805- 69.545059j
3,1,2,0.036990,0.011739,0.000000+0.000093j,0.036990+0.011739j,24.560562- 7.794692j
4,2,3,0.075000,0.216250,0.000000+0.000000j,0.075000+0.216250j,1.431597- 4.127770j
5,3,4,0.049545,0.014620,0.000000+0.000111j,0.049545+0.014620j,18.566948- 5.478833j


In [53]:
df_pu.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype     
---  ------    --------------  -----     
 0   from_bus  6 non-null      int64     
 1   to_bus    6 non-null      int64     
 2   r_pu      6 non-null      float64   
 3   x_pu      6 non-null      float64   
 4   b_pu      6 non-null      complex128
 5   z_pu      6 non-null      complex128
 6   y_pu      6 non-null      complex128
dtypes: complex128(3), float64(2), int64(2)
memory usage: 700.0 bytes


It is generally better to check the difference with the initial dataset:

In [54]:
net["line"]

,name,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,max_i_ka,parallel,type,r0_ohm_per_km,x0_ohm_per_km,c0_nf_per_km,g_us_per_km,df,std_type,in_service
0,line_0,0,1,10.0,0.006000,0.017300,0.0,0.627,1,None,0.0,0.0,0.0,0.0,1.0,None,True
1,line_1,0,3,15.0,0.009909,0.002924,27.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True
2,line_2,0,4,7.0,0.001190,0.000510,5.5,0.380,1,None,0.0,0.0,0.0,0.0,1.0,None,True
3,line_3,1,2,9.0,0.006576,0.002087,20.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True
4,line_4,2,3,20.0,0.006000,0.017300,0.0,0.627,1,None,0.0,0.0,0.0,0.0,1.0,None,True
5,line_5,3,4,8.0,0.009909,0.002924,27.5,0.290,1,None,0.0,0.0,0.0,0.0,1.0,None,True


Next, we are going to define the setpoints to run the load flow algorithm.

### Define setpoints

This function helps defining the power and voltage setpoints for you pandapower network created above.

In [55]:
def define_setpoints(net: pp.pandapowerNet, S_base: float)  -> tuple[np.array]:
    """Define power and voltage setpoints based on a pandapower network.

    Parameters
    ----------
    net : pandapowerNet
        The pandapower network from which setpoints are to be defined.
    S_base : float
        The system base apparent power (S_base) in MVA for power setpoints.    
    
    Returns
    -------
    tuple of numpy.ndarray
        A tuple containing the following arrays:
        - Psp (numpy.ndarray): Active power setpoints in per unit (p.u.).
        - Qsp (numpy.ndarray): Reactive power setpoints in per unit (p.u.).
        - Usp (numpy.ndarray): Voltage setpoints in complex per unit (p.u.).
    
    """
    # Define voltages starting points using external grid modul and angle values
    Usp = np.array([polar2z(*net.ext_grid.loc[0, ["vm_pu", "va_degree"]].values)]*net.bus.shape[0])
    # Initialize setpoints
    Psp =  np.zeros(net.bus.shape[0])
    Qsp =  np.zeros(net.bus.shape[0])
    # Power flow algorithm use generator sign convention.
    # Loads and PV power setpoints have to be converted from MW to p.u.
    if not net.load.empty:
        # First group every load connected to the same bus and sum up their corresponding active and reactive power
        loads = pd.concat([net.bus, net.load.groupby("bus")[["p_mw", "q_mvar"]].sum()*1e6/S_base], axis=1).fillna(0)
        Psp -= loads["p_mw"].values
        Qsp -= loads["q_mvar"].values
    if not net.sgen.empty:
        # First group every generators connected to the same bus and sum up their corresponding active and reactive power
        sgen = pd.concat([net.bus, net.sgen.groupby("bus")[["p_mw", "q_mvar"]].sum()*1e6/S_base], axis=1).fillna(0)
        Psp += sgen["p_mw"].values
        Qsp += sgen["q_mvar"].values
    return Psp, Qsp, Usp

### Compute admitance matrix

For the admittance matrix theory, please refer to RHT RE04-5, slide 27.

$\left\lbrack \underline{\mathbf{I}} \right\rbrack =\left\lbrack \underline{\mathbf{Y}} \right\rbrack \cdot \left\lbrack \underline{\mathbf{U}} \right\rbrack \rightarrow \text{Current vector}_{\left(N \times 1\right)} = \text{Admitance matrix}_{\left(N \times N\right)} \cdot \text{Voltage vector}_{\left(N \times 1\right)}$

- The elements in the diagonal $\underline{Y_{ii}}$ represent the sum of all the admittances connected to the $i$-bus, with a *positive* sign.
- The elements not in the diagonal $\underline{Y_{ij}}$ correspond to the sum of every admittance directly connecting the buses together, with a *negative* sign.

The general mathematical expression follows:

$
\begin{aligned}
    \underline{Y_{ij}} &= 
    \begin{cases} 
         \mathrm{j}\,b_{i} + \displaystyle\sum_l\left(\underline{y_{l}} +  \mathrm{j}\,\dfrac{b_{l}}{2}\right) &\text{if } i = j \\
        -\underline{y_{ij}} &\text{if } i \neq j
    \end{cases}
\end{aligned}
$

where:
- $b_{i}$ is the susceptance connected to the bus $i$
- $\underline{y_{l}} = \dfrac{1}{r_{l} + \mathrm{j}\,x_{l}}$ is the longitudinal admittance of lines connected to the bus $i$
- $b_{l}$ is the transversal susceptances of lines connected to the bus $i$
- $\underline{y_{ij}}$ is the longitudinal admittance of the line connected beteween bus $i$ and $j$


**Note**: if $\left\lbrack \underline{Y} \right\rbrack$ is non-singular, we can write: $\left\lbrack \underline{\mathbf{Z}} \right\rbrack ={\left\lbrack \underline{\mathbf{Y}} \right\rbrack }^{-1}$ and we have the following matrix relationship: $\left\lbrack \underline{\mathit{\mathbf{U}}} \right\rbrack =\left\lbrack \underline{\mathit{\mathbf{Z}}} \right\rbrack \cdot \left\lbrack \underline{\mathbf{I}} \right\rbrack$. With the $\left\lbrack \underline{\mathbf{Z}} \right\rbrack$ the nodal impedance matrix.

In [56]:
def compute_admittance_matrix(line_pu: pd.DataFrame) -> np.array:
    """Compute the admittance matrix for a power system based on line data.
    
    Parameters
    ----------
    line_pu : pd.DataFrame
        A DataFrame containing line data in per unit (p.u.) system.

    Returns
    -------
    np.array
        A complex-valued numpy array representing the admittance matrix of the power system.
    """
    # Number of lines
    Nl = line_pu.shape[0]
    # Number of bus
    Nb = pd.unique(line_pu[['from_bus', 'to_bus']].values.ravel('K')).shape[0]
    # Admittance matrix initialization
    Y_bus= np.zeros([Nb, Nb], dtype=np.complex128)
    # Compute admittance matrix 
    # Non-diagonal elements
    for l in range(Nl):
        # Find bus i and j connected to line l
        i, j = line_pu.at[l, "from_bus"], line_pu.at[l, "to_bus"]
        # Calculate admittance matrix non-diagonal elements i-j and j-i
        Y_bus[i, j] = - line_pu.at[l, "y_pu"]
        Y_bus[j, i] = - line_pu.at[l, "y_pu"]
    # Diagonal elements
    for i in range(Nb):
        # Find every line connected to bus i
        line_connected = line_pu[(line_pu[["from_bus", "to_bus"]] == i).any(axis=1)]
        # line_pu[(line_pu["from_bus"] == 1) | (line_pu["to_bus"] == 1)]
        # Calculate admittance matrix diagonal element i
        Y_bus[i, i] = line_connected["y_pu"].sum() + line_connected["b_pu"].sum()/2
        
    return Y_bus

**Note**: The code `Nb = pd.unique(line_pu[['from_bus', 'to_bus']].values.ravel('K')).shape[0]` is used to count the number of unique buses in the `line_pu` DataFrame. The `line_pu` DataFrame is assumed to contain two columns: `from_bus` and `to_bus`. These columns represent the start and end buses of each line in the power grid.

For the other part of the code, we just built the admittance matrix developed during the course.

### Compute power flow

For a $N$–buses grid, power flow is calculated for every bus $i$: $\underline{S_{i}} = \underline{U_{i}}\cdot\underline{I_{i}}^{*}$.

By using generalized ohm law and admittance matrix, currents are express as follows:

$
\begin{aligned}
    \left\lbrack \underline{\mathbf{I}} \right\rbrack & = 
    \left\lbrack \underline{\mathbf{Y}} \right\rbrack \cdot \left\lbrack \underline{\mathbf{U}} \right\rbrack \\[0.5em]
    \begin{pmatrix} \underline{I_{1}} \\ \vdots \\ \underline{I_{N}} \end{pmatrix} & = 
    \begin{pmatrix} \underline{Y_{11}} & \cdots &  \underline{Y_{1N}} \\ \vdots & \ddots & \vdots \\ \underline{Y_{N1}} & \cdots &  \underline{Y_{NN}} \end{pmatrix} \cdot
    \begin{pmatrix} \underline{U_{1}} \\ \vdots \\ \underline{U_{N}} \end{pmatrix} \\[0.5em]
    \Rightarrow \underline{I_{i}} & = \sum_{h=1}^N\underline{Y_{ih}} \cdot \underline{U_{h}}.
\end{aligned}
$

Therefore, the $i$-bus' active and reactive powers are given by:

$
\begin{aligned}
    \underline{S_{i}} &= \underline{U_{i}}\cdot\left(\displaystyle\sum_{h=1}^N\underline{Y_{ih}}\cdot\underline{U_{h}}\right)^{*} = P_i + j\, Q_i \\
    & \Rightarrow 
    \begin{cases} 
    P_{i} &= \displaystyle\sum_{h=1}^N U_{i} U_{h} \cdot \lbrack G_{ih} \cos\left(\theta_{i}-\theta_{h}\right) + B_{ih} \sin\left(\theta_{i}-\theta_{h}\right)\rbrack \\
    Q_{i} &= \displaystyle\sum_{h=1}^N U_{i} U_{h} \cdot \lbrack G_{ih} \sin\left(\theta_{i}-\theta_{h}\right) - B_{ih} \cos\left(\theta_{i}-\theta_{h}\right)\rbrack
    \end{cases}
\end{aligned}
$

with $G_{ih}$ and $B_{ih}$ the real and imaginary parts of the element $\underline{Y_{ih}}$ of the admittance matrix respectively.

In [57]:
def update_bus_power(Y_bus: np.array, U: np.array) -> tuple[np.array, np.array]:
    """Update the active and reactive power injections at each bus based on the admittance matrix and voltage magnitudes and angles.
    
    Parameters
    ----------
    Y_bus : np.array
        The complex-valued admittance matrix of the power system.
    U : np.array
        An array of complex-valued bus voltages.

    Returns
    -------
    tuple of np.array
        A tuple containing the following arrays:
        - P (np.array): Array of active power injections at each bus.
        - Q (np.array): Array of reactive power injections at each bus.
    """
    # Number of buses
    Nb = Y_bus.shape[0]
    # Decompose complex voltages into polar coordinates
    U_abs, U_angle = np.abs(U), np.angle(U)
    # Decompose complex admittance matrix into conductance and susceptance matrix
    G_bus, B_bus = np.real(Y_bus), np.imag(Y_bus)
    # Initialize outputs
    P,  Q  = np.zeros(Nb), np.zeros(Nb)
    # Calculate bus powers 
    for i in range(Nb):
        for j in range(Nb):
            angle = U_angle[i] - U_angle[j]
            P[i] += U_abs[i] * U_abs[j] * (G_bus[i, j] * np.cos(angle) + B_bus[i, j] * np.sin(angle))
            Q[i] += U_abs[i] * U_abs[j] * (G_bus[i, j] * np.sin(angle) - B_bus[i, j] * np.cos(angle))
    return P, Q

### Build Jacobian matrix

For the Newton-Raphson theory, please refer to RHT RE04-5, slides 30–35.

The Jacobian Matrix $\mathbf{J}$ contains the partial derivatives of the equations of the injected $P$ and $Q$ with respect to the state variables. At the $k$-iteration

$
\begin{aligned}
\mathbf{J}^{k}\left(\mathbf{x}\right) &= 
    \begin{pmatrix}
    \dfrac{\partial\,\mathbf{P}}{\partial\,\mathbf{\theta}}\left(\mathbf{\theta}^{k},{\mathbf{U}}^{k}\right) & \dfrac{\partial\,\mathbf{P}}{\partial\,\mathbf{U}}\left(\mathbf{\theta}^{k},{\mathbf{U}}^{k}\right) \\[1em]
    \dfrac{\partial\,\mathbf{Q}}{\partial\,\theta}\left(\mathbf{\theta}^{k},{\mathbf{U}}^{k}\right) & \dfrac{\partial\,\mathbf{Q}}{\partial\,\mathbf{U}}\left(\mathbf{\theta}^{k},{\mathbf{U}}^{k}\right)
    \end{pmatrix}\\[1em]
\text{where}: \\[1em]
\dfrac{\partial\,P_{i}}{\partial\,\theta_{j}} &=
    \begin{cases} \displaystyle\sum_{h=1;\,h \neq i}^{N} U_{i}\,U_{h}\,Y_{ih}\sin(\theta_{h}-\theta_{i}+\gamma_{ih}), \quad\quad &\text{ if } ~i=j \\
    - U_{i}\,U_{j}\,Y_{ij}\sin(\theta_{j}-\theta_{i}+\gamma_{ij}), \quad\quad &\text{ if } ~i\neq j 
    \end{cases} \\[0.5em]
\dfrac{\partial\,Q_{i}}{\partial\,\theta_{j}} &= 
    \begin{cases}\displaystyle\sum_{h=1;\,h \neq i}^{N} U_{i}\,U_{h}\,Y_{ih}\cos(\theta_{h}-\theta_{i}+\gamma_{ih}), \quad\quad &\text{ if } ~i=j \\
    -U_{i}\,U_{j}\,Y_{ij}\cos(\theta_{j}-\theta_{i}+\gamma_{ij}), \quad\quad &\text{ if } ~i\neq j 
    \end{cases}\\[2.5em]
\dfrac{\partial\,P_{i}}{\partial\,U_{j}} &= 
    \begin{cases} 2\,U_{i}\,Y_{ii}\cos(\gamma_{ii}) + \displaystyle\sum_{h=1;\,h \neq i}^{N} U_{h}\,Y_{ih}\cos(\theta_{h}-\theta_{i}+\gamma_{ih}), \quad\quad &\text{ if } ~i=j \\
    U_{i}\,Y_{ij}\sin(\theta_{j}-\theta_{i}+\gamma_{ij}), \quad\quad &\text{ if } ~i\neq j 
    \end{cases} \\[0.5em]
\dfrac{\partial\,Q_{i}}{\partial\,U_{j}} &= 
\begin{cases} - 2\,U_{i}\,Y_{ii}\sin(\gamma_{ii}) - \displaystyle\sum_{h=1;\,h \neq i}^{N} U_{h}\,Y_{ih}\sin(\theta_{h}-\theta_{i}+\gamma_{ih}), \quad\quad &\text{ if } ~i=j \\ 
-U_{i}\,Y_{ij}\sin(\theta_{j}-\theta_{i}+\gamma_{ij}), \quad\quad &\text{ if } ~i\neq j
\end{cases}
\end{aligned}
$

Where $\gamma_{ih}$ is the admitance matrix $ih$-element's angle.

In [58]:
def jacobian_matrix(Y_bus: np.array, U:np.array, slack_bus: int) -> np.array:
    """Calculate the Jacobian matrix for a power system based on admittance matrix and voltage values.

    Parameters
    ----------
    Y_bus : np.array
        A complex-valued admittance matrix of the power system.
    U : np.array
        An array of complex-valued bus voltages.
    slack_bus : int
        The index of the slack bus.

    Returns
    -------
    np.array
        The Jacobian matrix of the power system, which relates changes in bus voltages to changes in active and reactive power injections.
    """
    # Number of buses
    Nb = Y_bus.shape[0]
    # Initialize active and reactive power partial differential depending on voltages modulus and angle.
    dP_dU = np.zeros([Nb, Nb])
    dP_dangle = np.zeros([Nb, Nb])
    dQ_dU = np.zeros([Nb, Nb])
    dQ_dangle = np.zeros([Nb, Nb])
    
    # Decompose complex voltages into polar coordinates
    U_abs, U_angle = np.abs(U), np.angle(U)
    # Decompose complex admittance matrix into polar coordinates
    Y_abs, Y_angle = np.abs(Y_bus), np.angle(Y_bus)

    # Calculate active and reactive power partial differential depending on voltages modulus and angle.
    for i in range(Nb):
        dP_dU[i, i] = 2 * Y_abs[i, i] * U_abs[i] * np.cos(Y_angle[i, i])
        dQ_dU[i, i] = -2 * Y_abs[i, i] * U_abs[i] * np.sin(Y_angle[i, i])
        for j in range(Nb):
            if i != j:
                # Calculate angle
                angle = Y_angle[i, j] + U_angle[j] - U_angle[i]
        
                dP_dangle[i, i] += Y_abs[i, j] * U_abs[i] * U_abs[j] * np.sin(angle)
                dP_dU[i, i] +=  Y_abs[i, j] * U_abs[j] * np.cos(angle)
                dQ_dangle[i, i] += Y_abs[i, j] * U_abs[i] * U_abs[j] * np.cos(angle)
                dQ_dU[i, i] -= Y_abs[i, j] * U_abs[j] * np.sin(angle)
         
                dP_dangle[i, j] = -Y_abs[i, j] * U_abs[i] * U_abs[j]*np.sin(angle)
                dP_dU[i, j] = Y_abs[i, j] * U_abs[i] * np.cos(angle)
                dQ_dangle[i, j] = -Y_abs[i, j] * U_abs[i] * U_abs[j]* np.cos(angle)
                dQ_dU[i, j] = -Y_abs[i, j] * U_abs[i] * np.sin(angle)
                        
    # Remove slack nodes
    dP_dU = remove_slack_bus(dP_dU, slack_bus)
    dP_dangle = remove_slack_bus(dP_dangle, slack_bus)
    dQ_dU = remove_slack_bus(dQ_dU, slack_bus)
    dQ_dangle = remove_slack_bus(dQ_dangle, slack_bus)
    # Create Jacobian matrix
    J = np.concatenate([
            np.concatenate([dP_dU, dP_dangle], axis= 1),
            np.concatenate([dQ_dU, dQ_dangle], axis= 1)
        ])
    print("\nJacobian Matrix:\n", pd.DataFrame(J).to_string())
    return J

### Update voltages

A N-bus grid is usually composed by:
- 1 slack bus 
- M-1 PV bus
- N-M PQ bus

Newton-Raphson methode will update voltage iteratively:

$
\begin{aligned}
    \mathbf{x}^{k+1} &= \left(\mathbf{J}^{k}\right)^{-1} \cdot \mathbf{f}\left(\mathbf{x}^{k}\right) + \mathbf{x}^{k}\\
    \text{where:} & \\
    \mathbf{x}^{k} &= 
    \begin{pmatrix} 
        \theta_{2}^{k} \\ \vdots \\ \theta_{N}^{k} \\ U_{M+1}^{k} \\ \vdots \\ U_{N}^{k} 
    \end{pmatrix} \text{;} \;  
    \mathbf{f}\left(\mathbf{x}^{k}\right) = 
    \begin{pmatrix} 
        P_{2}\left(\mathbf{x}^{k}\right) - P_{2}^{SP} \\ 
        \vdots \\
        P_{N}\left(\mathbf{x}^{k}\right) - P_{N}^{SP} \\ 
        Q_{M+1}\left(\mathbf{x}^{k}\right) - Q_{M+1}^{SP} \\ 
        \vdots \\
        Q_{N}\left(\mathbf{x}^{k}\right) - Q_{N}^{SP}
    \end{pmatrix}.
\end{aligned}
$

where:

- $P_{i}^{SP}$ is the bus $i$ active power setpoint
- $Q_{i}^{SP}$ is the bus $i$ reactive power setpoint


In [59]:
def update_voltages(dP: np.array, dQ: np.array, U: np.array, J: np.array, slack_bus: int) -> np.array:
    """Update the bus voltages based on changes in active and reactive power injections and the Jacobian matrix.
    
    Parameters
    ----------
    dP : np.array
        An array of changes in active power injections at each bus.
    dQ : np.array
        An array of changes in reactive power injections at each bus.
    U : np.array
        An array of complex-valued bus voltages.
    J : np.array
        The Jacobian matrix relating changes in bus voltages to changes in active and reactive power injections.
    slack_bus : int
        The index of the slack bus.

    Returns
    -------
    np.array
        An array representing the updated complex bus voltages after the power flow calculations.
    """
    # Number of buses
    Nb = U.shape[0]
    # Calculate state variable delta
    dU = np.matmul(inv(J), np.concatenate([dP, dQ]))
    # Update voltages in polars form
    U_abs_new = np.abs(U) + np.insert(dU[:Nb-1],slack_bus,0)
    U_angle_new = np.angle(U) + np.insert(dU[Nb-1:],slack_bus,0)
    # Build complex voltages
    U_new = polar2z(module=U_abs_new, angle=U_angle_new)
    print("\nBus voltages:\n", pd.DataFrame(U_new).to_string(), "\n")
    return U_new

### Process results

Use generalized ohm law to compute lines currents ant powers flows

$
\begin{aligned}
    \underline{I_{ij}} &= \left(\underline{U_{i}} - \underline{U_{j}}\right) \cdot y_{ij} + \underline{U_{i}}  \cdot b_{ij}\\
    \underline{I_{ji}} &= \left(\underline{U_{j}} - \underline{U_{i}}\right) \cdot y_{ij} + \underline{U_{j}}  \cdot b_{ij}
\end{aligned}
$

In [60]:
def process_results(line_pu: pd.DataFrame, U: np.array, P: np.array, Q: np.array, V_base: float, S_base:float) -> pd.DataFrame:
    """Process power flow results and calculate line and bus-related parameters.

    Parameters
    ----------
    line_pu : pd.DataFrame
        A DataFrame containing line data in per unit (p.u.) system.
    U : np.array
        An array of complex-valued bus voltages.
    P : np.array
        An array of active power injections at each bus.
    Q : np.array
        An array of reactive power injections at each bus.
    V_base : float
        The base voltage of the system in kV.
    S_base : float
        The system base apparent power (S_base) in MVA for power setpoints.

    Returns
    -------
    pd.DataFrame, pd.DataFrame
        Two DataFrames containing the following results:
        - line_results (pd.DataFrame): Line-related parameters including active power (p_from_mw, p_to_mw), 
        reactive power (q_from_mvar, q_to_mvar), current (i_from_ka, i_to_ka) in kA.
        - bus_result (pd.DataFrame): Bus-related parameters including voltage magnitude (vm_pu), 
        voltage angle (va_degree), active power (p_mw), and reactive power (q_mvar).
    """
    # Initialize line results DataFrame
    line_results = pd.DataFrame(
        index=line_pu.index, columns=["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar", "i_from_ka", "i_to_ka"]
    )
    # Initialize bus results DataFrame
    bus_results = pd.DataFrame()
    # Calculate base current
    I_base = S_base/(np.sqrt(3)*V_base)
    
    for i in line_pu.index:
        # Find from_bus
        from_bus = line_pu.at[i, "from_bus"]
        # Find to_bus
        to_bus = line_pu.at[i, "to_bus"]
        # Find longitudinal admittance
        y_line = line_pu.at[i, "y_pu"]
        # Find transversal admittance
        b_line = line_pu.at[i, "b_pu"]
        # Calculate line current in pu
        i_from = ((U[from_bus] - U[to_bus]) * y_line + U[from_bus] * b_line/2)
        i_to = ((U[to_bus] - U[from_bus]) * y_line + U[to_bus] * b_line/2)
        # Calculate line apparent power in pu  
        s_from = U[from_bus] * np.conjugate(i_from)
        s_to = U[to_bus] * np.conjugate(i_to)
        # Create line results in pandapower format  
        line_results.loc[i, :]  = [
            np.real(s_from)*S_base/1e6, np.imag(s_from)*S_base/1e6, np.real(s_to)*S_base/1e6, np.imag(s_to)*S_base/1e6, 
            np.abs(i_from) * I_base/1e3, np.abs(i_to) * I_base/1e3
        ]
    # Create bus results in pandapower format     ]
    bus_results[["vm_pu", "va_degree", "p_mw", "q_mvar"]] = np.array([
        np.abs(U), np.angle(U)*180/np.pi, -P*S_base/1e6, -Q*S_base/1e6
    ]).transpose()    
    return line_results, bus_results

## Power Flow algorithm in a nutshell

1. Define bus types.
2. Define power and voltage setpoints.
3. Build admittance matrix.
4. Initialize voltages
5. Update bus powers
6. Calculate power mismatches: $ h\left(x^{k}\right) = \begin{bmatrix} \Delta P_{i} \left(x^{k}\right), \, \Delta Q_{i}\left(x^{k}\right) \end{bmatrix}^{T}$
7. Build Jacobian matrix.
8. Update voltage phasors in each bus.
9. Perform stop test $ \begin{cases} \max \lvert h\left(x^{k}\right) \rvert \leq \epsilon \Rightarrow &\text{the algorithm stops}. \\ \max\lvert h\left(x^{k}\right)\rvert > \epsilon \Rightarrow &\text{go to step 5}\ldots \end{cases}$


In [61]:
# Define set points in pu
Psp, Qsp, Usp = define_setpoints(net = net, S_base=S_base)
# Define slack bus index
slack_bus = net.ext_grid.at[0, "bus"]
# Convert line resistor and inductor into admittance complex parameter in p.u.
line_pu = convert_line_data_in_pu(net=net, Z_base=Z_base, f_base=f_base)
line_pu

,from_bus,to_bus,r_pu,x_pu,b_pu,z_pu,y_pu
0,0,1,0.037500,0.108125,0.000000+0.000000j,0.037500+0.108125j,2.863193- 8.255540j
1,0,3,0.092897,0.027412,0.000000+0.000207j,0.092897+0.027412j,9.902372- 2.922044j
2,0,4,0.005206,0.002231,0.000000+0.000019j,0.005206+0.002231j,162.271805- 69.545059j
3,1,2,0.036990,0.011739,0.000000+0.000093j,0.036990+0.011739j,24.560562- 7.794692j
4,2,3,0.075000,0.216250,0.000000+0.000000j,0.075000+0.216250j,1.431597- 4.127770j
5,3,4,0.049545,0.014620,0.000000+0.000111j,0.049545+0.014620j,18.566948- 5.478833j


In [62]:
# Compute admittance matrix
Y_bus = compute_admittance_matrix(line_pu=line_pu)
# Initialization
U = Usp
for counter in range(iter_max):
    print('Iteration {}'.format(counter + 1))
    # Update powers
    P, Q = update_bus_power(Y_bus=Y_bus, U=U)
    # Calculate power mismatches
    dP = remove_slack_bus(Psp-P, slack_bus)
    dQ = remove_slack_bus(Qsp-Q, slack_bus)
    # build Jacobian matrix
    J = jacobian_matrix(Y_bus=Y_bus, U=U, slack_bus=slack_bus)
    # Update voltages
    U = update_voltages(dP=dP, dQ=dQ, U=U, J=J,slack_bus=slack_bus)
    # Perform stop test
    tol = max(max(abs(dP)), max(abs(dQ)))
    if tol < tol_limit:
        print('Tolerance achieved: {}'.format(tol))
        print('Finished at {} iterations!'.format(counter + 1))
        break
# Compute lines and bus results
line_results, bus_results = process_results(line_pu=line_pu, U=U, P=P, Q=Q, V_base=V_base, S_base=S_base)

Iteration 1

Jacobian Matrix:
            0          1          2           3          4          5          6           7
0  27.423755 -24.560562   0.000000    0.000000  16.050232  -7.794692  -0.000000   -0.000000
1 -24.560562  25.992159  -1.431597    0.000000  -7.794692  11.922462  -4.127770   -0.000000
2   0.000000  -1.431597  29.900917  -18.566948  -0.000000  -4.127770  12.528647   -5.478833
3   0.000000   0.000000 -18.566948  180.838754  -0.000000  -0.000000  -5.478833   75.023893
4  16.050139  -7.794692  -0.000000   -0.000000 -27.423755  24.560562  -0.000000   -0.000000
5  -7.794692  11.922369  -4.127770   -0.000000  24.560562 -25.992159   1.431597   -0.000000
6  -0.000000  -4.127770  12.528329   -5.478833  -0.000000   1.431597 -29.900917   18.566948
7  -0.000000  -0.000000  -5.478833   75.023763  -0.000000  -0.000000  18.566948 -180.838754

Bus voltages:
                     0
0  1.000000+0.000000j
1  0.974480-0.030832j
2  0.942671-0.036744j
3  0.963228-0.005139j
4  1.003418+0.0

## Compare to **panda**power

Now, it is interesting to check the difference with the classical functions of **panda**power.

In [1]:
pp.runpp(net)
bus_pp_results = net.res_bus[["p_mw","q_mvar", "vm_pu", "va_degree"]]
line_pp_results = net.res_line[["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar", "i_from_ka", "i_to_ka"]]
line_error =(line_results-line_pp_results)/line_results
bus_error=(bus_results-bus_pp_results)/bus_results

print("Bus error:\n" + bus_error.abs().max().to_string() + "\n")
print("Line error:\n" + line_error.abs().max().to_string())

NameError: name 'pp' is not defined

## Plot the 5-mesh power grid

In [64]:
import pp_heig_plot as pp_plot
import pp_heig_simulation as pp_sim
from datetime import time

Once again we can check the topology of the grid we are working on:

In [65]:
pp_plot.plot_power_network(net=net, plot_title="5-mesh power grid example", filename="5-mesh_grid_example")

We can visualize the result using the following functions:

In [66]:
pp_plot.plot_powerflow_result(net=net, plot_title="5-mesh power grid results", filename="5-mesh_grid_pp_result")
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-0.00888,-0.034823
1,0.971924,-2.012888,-0.05000,0.005000
2,0.938415,-2.477066,0.10000,0.010000
3,0.961182,-0.304387,0.10000,0.010000
4,1.003188,0.129512,-0.15000,0.005000
